# Imports

In [47]:
import pandas as pd
import json
import sys
import os
import plotly.express as px
from tqdm import tqdm
import numpy as np
sys.path.append('../..')
from scipy.sparse import csgraph, csr_matrix
from definitions import DATA_DIR, ROOT_DIR, KGS_DIR
from functools import reduce
from plotly.graph_objs import Figure
import plotly.io as pio

In [2]:
FIGURES_DIR = os.path.join(ROOT_DIR, 'assets', 'graph_statistics')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [3]:
pio.templates.default = "seaborn"

In [4]:
def save_fig(fig : Figure, filename : str) -> None:
    fig.write_image(os.path.join(FIGURES_DIR, filename))

In [5]:
def show(fig : Figure) -> None:
    fig.update_layout(font_size = 12)
    fig.show()

# Utils

In [49]:
def load_graph(
    corpus : str,
    stage : str,
    predicate_min_occ : int = 30,
    entity_min_occ : int = 2
) -> pd.DataFrame: 
    
    path = os.path.join(KGS_DIR, corpus, stage, 'data.jsonl')

    kg = []

    with open(path, "r") as f:

        with tqdm(total=os.path.getsize(path)) as pbar:
            
            for line in f:
                record = json.loads(line)
                kg.append(record)
                pbar.update(len(line))

    kg = pd.DataFrame(kg)

    predicates = kg['predicate'].value_counts()
    entities = pd.Series(data = kg['head'].to_list() + kg['tail'].to_list()).value_counts()

    predicates = predicates.index[predicates >= predicate_min_occ]
    entities = entities.index[entities >= entity_min_occ]

    mask = (
        kg['head'].isin(entities) &
        kg['tail'].isin(entities) &
        kg['predicate'].isin(predicates)
    )

    return kg[mask]

In [8]:
def melt_df(kg : pd.DataFrame) -> pd.DataFrame:

    melted = []

    for i,(_,row) in tqdm(enumerate(kg.iterrows()), total=len(kg)):
                        
        for key in ["head","tail"]:

            melted.append({
                "triple_id" : 2 * i + int(key == "head"),
                "entity" : row[key],
                "predicate" : row["predicate"],
                "head" : key == "head",
                "tail" : key == "tail",
                "document" : row['sentence']['document']['id'] if row['sentence'] is not np.nan else None
            })

    melted = pd.DataFrame(melted)

    return melted

In [9]:
def get_components(df : pd.DataFrame) -> pd.Series:

    entities = list(set(df['head'].to_list() + df['tail'].to_list()))
    entities_map = { e : i for i,e in enumerate(entities) }
    head = df['head'].map(entities_map).to_numpy()
    tail = df['tail'].map(entities_map).to_numpy()

    N = len(entities)
    indices = np.stack([head, tail]).transpose()
    data = np.ones(len(df))

    adj = csr_matrix(
        (data, (indices[:, 0], indices[:, 1])),
        shape=(N, N),
        dtype=int
    )

    _, labels = csgraph.connected_components(adj, directed=False)

    labels = pd.Series(
        index=entities,
        data=labels
    )

    return labels

In [64]:
def describe_graph(kg : pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.Series]:

    kg['doc_id'] = kg['id'].str.split('_').apply(lambda x : x[0])
    kg_doc = kg.drop_duplicates(subset=['head', 'predicate', 'tail', 'doc_id'])
    kg_trip = kg.drop_duplicates(subset=['head', 'predicate', 'tail'])

    ### Metl-Graph
    melted = melt_df(kg)
    melted = melted[~melted['document'].isna()]

    ### Node-Level
    print("Node level")
    node_level : list[pd.Series] = [
        melted['document'].groupby(melted['entity']).agg(lambda x : len(set(x))).rename('number-of-documents'),
    ]

    melted = melted.drop_duplicates(subset=["triple_id"])

    node_level.extend([
        melted['head'].groupby(melted['entity']).sum().rename('out-degree'),
        melted['tail'].groupby(melted['entity']).sum().rename('in-degree'),
        melted['entity'].value_counts().rename('degree'),
    ])

    node_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        node_level,
        pd.DataFrame(index = node_level[0].index)
    )

    ### Relation-Level
    print("Predicate Level")
    relation_level : list[pd.Series] = [
        kg_trip['predicate'].value_counts().rename('number-of-triples')
    ]
    
    relation_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        relation_level,
        pd.DataFrame(index = relation_level[0].index)
    )

    ### Document Level
    print("Document Level")
    document_level : list[pd.Series] = [
        kg_doc['doc_id'].value_counts().rename('number-of-triples'),
        melted['entity'].groupby(by = melted['document']).agg(lambda x : len(set(x))).rename('number-of-entities')
    ]

    document_level : list[pd.Series] = reduce(
        lambda a,b : a.join(b.to_frame()),
        document_level,
        pd.DataFrame(index = document_level[0].index)
    )
    

    ### Components
    print("Connected Componenets")
    components = get_components(kg)
    component_level = [
        components.value_counts().rename('number-of-entities'),
        kg_trip['head'].map(components).value_counts().rename('number-of-triples')
    ]

    component_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        component_level,
        pd.DataFrame(index = component_level[0].index)
    )

    ### Graph-Level
    print("Graph Level")
    graph_level = dict()

    graph_level['Number of Triples'] = len(kg_trip)
    graph_level['Number of Entities'] = melted['entity'].nunique()
    graph_level['Number of Predicates'] = melted['predicate'].nunique()

    graph_level['Avg. In-Degree'] = node_level['in-degree'].mean()
    graph_level['Avg. Out-Degree'] = node_level['out-degree'].mean()
    graph_level['Avg. Degree'] = node_level['degree'].mean()
    graph_level['Avg. Documents / Entity'] = node_level['number-of-documents'].mean()

    graph_level['Avg. Triples / Predicate'] = relation_level['number-of-triples'].mean()
    graph_level['Avg. Triples / Document'] = document_level['number-of-triples'].mean()
    graph_level['Avg. Entity / Document'] = document_level['number-of-entities'].mean()

    graph_level['Largest Connected Component (Number of Entities)'] = component_level['number-of-entities'].max()
    graph_level['Largest Connected Component (Number of Triples)'] = component_level['number-of-triples'].max()
    graph_level['Number Of Connected Components'] = len(component_level)

    graph_level = pd.Series(graph_level)

    return node_level, relation_level, document_level, component_level, graph_level

# Load the data

In [50]:
kg = load_graph(
    'health-care', 
    'entities_mapping', 
    entity_min_occ = 0, 
    predicate_min_occ = 0
).dropna(axis=0)
# kg = kg.drop_duplicates(subset='id')
kg.head()

100%|██████████| 4529302128/4529302128 [00:41<00:00, 108916748.99it/s]


,id,sentence,head,predicate,tail,predicate_group
0,40315957_0_1,"{'id': '40315957_0', 'sentence': 'Unlike trime...",methylamine pathway metabolite,role in,diabetic kidney disease,role
1,40315957_1_0,"{'id': '40315957_1', 'sentence': 'We investiga...",circulating methylamine,associate with,progression of diabetic kidney disease,associated
2,40315957_2_0,"{'id': '40315957_2', 'sentence': 'We analyzed ...",european origin cohort,analyze in,asian origin cohort,analyzed
3,40315957_4_0,"{'id': '40315957_4', 'sentence': 'In DKD , KFR...",kfrt,be,primary outcome,was
4,40315957_5_0,"{'id': '40315957_5', 'sentence': 'Baseline bet...",mass spectrometry,measure,baseline betaine concentration,measured


In [53]:
mask = kg['sentence'].apply(lambda x : max([item['score'] for item in x['document']['pseudo_labels']['llm:deepseek-chat']]))
mask = mask / mask.max()
mask = mask.gt(0.5)
# mask = kg['sentence'].apply(lambda x : len(x['document']['labels']) > 0)

In [54]:
sub_kg = kg[mask]

In [65]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg)

/tmp/ipykernel_12542/3134409735.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kg['doc_id'] = kg['id'].str.split('_').apply(lambda x : x[0])
100%|██████████| 268625/268625 [00:12<00:00, 21074.20it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [67]:
sub_kg['doc_id'].nunique()

12060

In [68]:
graph_level

Number of Triples                                   240478.000000
Number of Entities                                  164331.000000
Number of Predicates                                 21556.000000
Avg. In-Degree                                           1.634658
Avg. Out-Degree                                          1.634658
Avg. Degree                                              3.269316
Avg. Documents / Entity                                  1.704925
Avg. Triples / Predicate                                11.155966
Avg. Triples / Document                                 20.956965
Avg. Entity / Document                                  23.231509
Largest Connected Component (Number of Entities)    131368.000000
Largest Connected Component (Number of Triples)     217448.000000
Number Of Connected Components                       11475.000000
dtype: float64

In [24]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg)

100%|██████████| 108357/108357 [00:05<00:00, 21552.36it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [25]:
graph_level

Number of Triples                                   108357.000000
Number of Entities                                  105830.000000
Number of Predicates                                 12882.000000
Avg. In-Degree                                           1.023878
Avg. Out-Degree                                          1.023878
Avg. Degree                                              2.047756
Avg. Documents / Entity                                  1.325069
Avg. Triples / Predicate                                 8.411504
Avg. Triples / Document                                 30.024106
Avg. Entity / Document                                  19.428096
Largest Connected Component (Number of Entities)     56275.000000
Largest Connected Component (Number of Triples)      72043.000000
Number Of Connected Components                       16854.000000
dtype: float64

In [69]:
components = get_components(sub_kg)

In [70]:
components = get_components(sub_kg)
largest_cc = components.value_counts().idxmax().item()
largest_cc_entities = components.index[components == largest_cc]
lcc_mask = sub_kg['head'].isin(largest_cc_entities) & sub_kg['tail'].isin(largest_cc_entities)
lcc_node_level, lcc_relation_level, lcc_document_level, lcc_component_level, lcc_graph_level = describe_graph(sub_kg[lcc_mask])

/tmp/ipykernel_12542/3134409735.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kg['doc_id'] = kg['id'].str.split('_').apply(lambda x : x[0])
100%|██████████| 243792/243792 [00:10<00:00, 22340.65it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [61]:
lcc_graph_level

Number of Triples                                   93723.000000
Number of Entities                                  62917.000000
Number of Predicates                                 3968.000000
Avg. In-Degree                                          1.547722
Avg. Out-Degree                                         1.547722
Avg. Degree                                             3.095443
Avg. Documents / Entity                                 1.879460
Avg. Triples / Predicate                               23.619708
Avg. Triples / Document                                11.400947
Avg. Entity / Document                                 16.450473
Largest Connected Component (Number of Entities)    62917.000000
Largest Connected Component (Number of Triples)     93723.000000
Number Of Connected Components                          1.000000
dtype: float64

In [62]:
robotics_mask = sub_kg['sentence'].apply(lambda x :  
    [item['score'] for item in x['document']['pseudo_labels']['llm:deepseek-chat'] if item['label'] == 'Robotics'][0]
)
robotics_mask = robotics_mask / robotics_mask.max()
robotics_mask = robotics_mask.gt(0.5)

In [63]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg[robotics_mask])

/tmp/ipykernel_8438/3458869208.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kg['doc_id'] = kg['id'].apply(str).apply(lambda x : x[:-2])
100%|██████████| 10665/10665 [00:00<00:00, 22778.04it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [64]:
graph_level

Number of Triples                                   9831.000000
Number of Entities                                  9034.000000
Number of Predicates                                1341.000000
Avg. In-Degree                                         1.180540
Avg. Out-Degree                                        1.180540
Avg. Degree                                            2.361080
Avg. Documents / Entity                                1.453952
Avg. Triples / Predicate                               7.331096
Avg. Triples / Document                               11.631881
Avg. Entity / Document                                17.058442
Largest Connected Component (Number of Entities)    5928.000000
Largest Connected Component (Number of Triples)     7599.000000
Number Of Connected Components                      1089.000000
dtype: float64

In [65]:
sub_kg = sub_kg[robotics_mask]
components = get_components(sub_kg)
largest_cc = components.value_counts().idxmax().item()
largest_cc_entities = components.index[components == largest_cc]
robotics_lcc_mask = sub_kg['head'].isin(largest_cc_entities) & sub_kg['tail'].isin(largest_cc_entities)
lcc_node_level, lcc_relation_level, lcc_document_level, lcc_component_level, lcc_graph_level = describe_graph(sub_kg[robotics_lcc_mask])

/tmp/ipykernel_8438/3458869208.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kg['doc_id'] = kg['id'].apply(str).apply(lambda x : x[:-2])
100%|██████████| 8376/8376 [00:00<00:00, 21949.07it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [66]:
lcc_graph_level

Number of Triples                                   7599.000000
Number of Entities                                  5928.000000
Number of Predicates                                1146.000000
Avg. In-Degree                                         1.412955
Avg. Out-Degree                                        1.412955
Avg. Degree                                            2.825911
Avg. Documents / Entity                                1.653677
Avg. Triples / Predicate                               6.630890
Avg. Triples / Document                                9.414371
Avg. Entity / Document                                13.053262
Largest Connected Component (Number of Entities)    5928.000000
Largest Connected Component (Number of Triples)     7599.000000
Number Of Connected Components                         1.000000
dtype: float64

# Visualization

## Node-Level 

- In-Degrees

In [20]:
fig = px.histogram(
    data_frame = node_level,
    x = "in-degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "In-Degree", yaxis_title = "Number of Entities", font_size=17)

save_fig(fig, 'in_degree_distribution.pdf')
show(fig)

- Out Degree

In [21]:
fig = px.histogram(
    data_frame = node_level,
    x = "out-degree",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = "Out-Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'out_degree_distribution.pdf')
show(fig)

- Degree

In [22]:
fig = px.histogram(
    data_frame = node_level,
    x = "degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'node_degree_distribution.pdf')
show(fig)

- Most frequent entities

In [23]:
fig = px.bar(
    data_frame = node_level['degree'].sort_values(ascending = False).reset_index()[:20][::-1],
    y = "entity",
    x = "degree",
    width = 800,
)

fig.update_layout(xaxis_title = "Number of Triplets", yaxis_title = "Entity")
save_fig(fig, 'top_10_most_frequent_entities.pdf')
show(fig)

- Predicates

In [24]:
fig = px.histogram(
    data_frame = relation_level.reset_index(),
    x = "number-of-triples",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = 'Relation Frequency', yaxis_title = "Number Of Relations")
save_fig(fig, 'relations_frequency_distribution.pdf')
show(fig)

In [25]:
fig = px.bar(
    data_frame = relation_level.sort_values(by = "number-of-triples", ascending = False).reset_index()[:20][::-1],
    y = "predicate",
    x = "number-of-triples",
    width = 800,
    log_x=True
)
fig.update_layout(xaxis_title = 'Number of Triplets', yaxis_title = "Predicate")
save_fig(fig, 'top_10_most_frequent_predicates.pdf')
show(fig)

In [33]:
kg[kg['sentence'].isna()]['predicate'].value_counts()

predicate
use                          4007
skosbroader be hyponym of    2774
include                       911
analyze                       885
base on                       494
produce                       484
solve                         287
improve                       258
acquire                       241
match                         141
provide                       140
support                       134
execute                       123
limit                          77
propose                        66
identifie                      37
transfer                       31
learn                          30
affect                         20
link                           17
classifie                      16
predict                        16
contribute to                  15
manage                         15
extract                        12
interact with                  10
outperform                      6
guide                           6
process                         5
caus